# Notebook 03 - From graphs to Hamiltonians

In [13]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.graphs import load_instances, json_to_graph, find_instance
from src.classical import bisection_bruteforce
from src.hamiltonians import max_cut_hamiltonian, generate_penalty_A, bisection_hamiltonian, check_balanced_bitstring, bisection_ground_states, count_zz_terms

instances = load_instances("../data/instances.json")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Max-Cut needs one ZZ term per edge, but the penalty $(\sum Z_i)^2$ couples every pair of qubits, so bisection always has n(n−1)/2 terms (15 at n = 6, 91 at n = 14) regardless of the graph

On sparse graphs this is up to 6.5x more terms (ring_14), and on hardware with limited connectivity every extra pair costs SWAP gates

This is why bisection circuits are much deeper on the QPU (Stage 5: ~3 CZ per pair; F5)

In [7]:
rows = []

for instance in instances:
    graph = json_to_graph(instance)
    n = instance["n"]

    H_mc = max_cut_hamiltonian(graph)
    H_b = bisection_hamiltonian(graph, generate_penalty_A(graph))

    row = {
        "name": instance["name"],
        "graph_type": instance["graph_type"],
        "n": n,
        "num_edges": graph.number_of_edges(),
        "zz_maxcut": count_zz_terms(H_mc),
        "zz_bisection": count_zz_terms(H_b),
        "all_pairs": n * (n - 1) // 2,
    }
    rows.append(row)

terms_table = pd.DataFrame(rows)
terms_table["ratio"] = terms_table["zz_bisection"] / terms_table["zz_maxcut"]
terms_table

,name,graph_type,n,num_edges,zz_maxcut,zz_bisection,all_pairs,ratio
0,ring_6,ring,6,6,6,15,15,2.500000
1,grid_6,grid,6,7,7,15,15,2.142857
2,regular3_6,regular3,6,9,9,15,15,1.666667
3,random_6,random,6,8,8,15,15,1.875000
4,complete_6,complete,6,15,15,15,15,1.000000
5,ring_8,ring,8,8,8,28,28,3.500000
6,grid_8,grid,8,10,10,28,28,2.800000
7,regular3_8,regular3,8,12,12,28,28,2.333333
8,random_8,random,8,15,15,28,28,1.866667
9,complete_8,complete,8,28,28,28,28,1.000000


In [8]:
terms_table.to_csv("../results/hamiltonian_terms.csv", index=False)

The theoretical bound (A > d_max/4) guarantees that every ground state is balanced

In practice it is very conservative: ring_8 keeps balanced ground states down to A = 0.05 (10x below the bound), random_10 down to A = 0.375 (4x below) and breaks only at 0.15

This project still uses the safe value A = (d_max + 1)/4: a smaller A would make the energy landscape less steep, but correctness on every graph matters more

In [9]:
for name in ["ring_8", "random_10"]:
    instance = find_instance(instances, name)
    graph = json_to_graph(instance)
    n = instance["n"]

    degrees = dict(graph.degree())
    threshold = max(degrees.values()) / 4
    optimum, _ = bisection_bruteforce(graph)

    print(name, "| threshold:", threshold, "| optimum:", optimum)

    for factor in [1.25, 1.0, 0.75, 0.5, 0.25, 0.1]:
        A = threshold * factor
        min_energy, ground_states = bisection_ground_states(graph, A)
        balanced = check_balanced_bitstring(ground_states, n)
        print("  A =", round(A, 3),
              "| min energy:", round(min_energy, 3),
              "| ground states:", len(ground_states),
              "| all balanced:", balanced)

ring_8 | threshold: 0.5 | optimum: 2
  A = 0.625 | min energy: 2.0 | ground states: 8 | all balanced: True
  A = 0.5 | min energy: 2.0 | ground states: 8 | all balanced: True
  A = 0.375 | min energy: 2.0 | ground states: 8 | all balanced: True
  A = 0.25 | min energy: 2.0 | ground states: 8 | all balanced: True
  A = 0.125 | min energy: 2.0 | ground states: 8 | all balanced: True
  A = 0.05 | min energy: 2.0 | ground states: 8 | all balanced: True
random_10 | threshold: 1.5 | optimum: 7
  A = 1.875 | min energy: 7.0 | ground states: 4 | all balanced: True
  A = 1.5 | min energy: 7.0 | ground states: 4 | all balanced: True
  A = 1.125 | min energy: 7.0 | ground states: 4 | all balanced: True
  A = 0.75 | min energy: 7.0 | ground states: 4 | all balanced: True
  A = 0.375 | min energy: 7.0 | ground states: 4 | all balanced: True
  A = 0.15 | min energy: 6.6 | ground states: 2 | all balanced: False
